# Original CNN Model for Binary Classification

This notebook contains the original CNN model with 682,721 parameters. This model demonstrates severe overfitting on the 8K sample dataset.


In [ ]:
import pandas as pd 
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt


In [ ]:
class BinaryClassificationCNN(nn.Module):
    """Original CNN model with many parameters - prone to overfitting on small datasets"""
    def __init__(self, input_channels: int = 4, sequence_length: int = 600):
        super(BinaryClassificationCNN, self).__init__()
        self.conv1 = nn.Conv1d(input_channels, 32, kernel_size=3, stride=1, padding=1)
        self.conv2 = nn.Conv1d(32, 64, kernel_size=5, stride=1, padding=2)
        self.conv3 = nn.Conv1d(64, 128, kernel_size=7, stride=1, padding=3)
        self.pool = nn.MaxPool1d(2)
        
        # Calculate the size after convolutions and pooling
        # After 3 pooling operations: sequence_length // (2^3) = sequence_length // 8
        self.final_length = sequence_length // 8
        self.fc1 = nn.Linear(128 * self.final_length, 64)
        self.fc2 = nn.Linear(64, 1)  # Single output for binary classification
        self.dropout = nn.Dropout(0.5)

    def forward(self, x):
        # x shape: [batch, length, channels] -> transpose to [batch, channels, length]
        x = x.transpose(1, 2)  # [batch, channels, length]
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))
        
        # Flatten for fully connected layers
        x = x.view(x.size(0), -1)
        x = self.dropout(F.relu(self.fc1(x)))
        x = torch.sigmoid(self.fc2(x))  # Sigmoid for binary classification
        
        return x
